# Week 2 Day 07 — Prompt Engineering

## Learning Objectives

- Build a reusable prompt library with 5–6 templates
- Compare zero-shot and few-shot prompting on the same task
- Force strict JSON output
- Parse and validate JSON safely
- Build a prompt A/B testing workflow
- Measure prompt performance and identify the winning prompt
- Understand common prompt failure modes and systematic iteration

## 1. Introduction to Prompt Engineering

Prompt engineering is the process of designing clear, specific, and reusable instructions for Large Language Models (LLMs).

Instead of simply asking a question, prompt engineering defines:

- Role
- Instruction
- Context
- Examples
- Output format
- Constraints

The goal is to make LLM responses more reliable, consistent, and useful for applications.

## 2. Prompt Library

A prompt library stores reusable prompt templates separately from application code.

Benefits:

- Prompts can be reused
- Prompts are easier to maintain
- Different prompt versions can be tested
- Prompt A/B testing becomes easier
- Application code stays cleaner

Our prompt library will contain:

1. Zero-shot sentiment classification
2. Few-shot sentiment classification
3. Summarization
4. General classification
5. JSON extraction
6. Prompt A/B testing

In [3]:
from pathlib import Path

PROMPTS_DIR = Path("prompts")

print("Prompt directory:")
print(PROMPTS_DIR.resolve())

print("\nDirectory exists:")
print(PROMPTS_DIR.exists())

Prompt directory:
D:\arun\ai-learning\week2\prompts

Directory exists:
True


In [4]:
for file in PROMPTS_DIR.iterdir():
    print(file.name)

classification.txt
few_shot_sentiment.txt
json_extraction.txt
prompt_ab_test.txt
summarization.txt
zero_shot_sentiment.txt


In [5]:
def load_prompt(filename):
    path = PROMPTS_DIR / filename

    with open(path, "r", encoding="utf-8") as file:
        return file.read()

In [6]:
zero_shot_prompt = load_prompt("zero_shot_sentiment.txt")

print(zero_shot_prompt)

You are a sentiment classification assistant.

Classify the following review as exactly one of:
POSITIVE
NEGATIVE
NEUTRAL

Review:
{review}

Return only the classification label.


## 3. Zero-Shot vs Few-Shot Prompting

We will compare two approaches for the same sentiment classification task.

### Zero-Shot

The model receives the task instructions but no examples.

### Few-Shot

The model receives examples showing how previous reviews should be classified.

We will use the same review for both approaches.

In [7]:
few_shot_prompt = load_prompt("few_shot_sentiment.txt")

print(few_shot_prompt)

You are a sentiment classification assistant.

Classify the following review as exactly one of:
POSITIVE
NEGATIVE
NEUTRAL

Examples:

Review:
"I absolutely loved this movie."
Classification:
POSITIVE

Review:
"The movie was terrible and boring."
Classification:
NEGATIVE

Review:
"The movie was okay, nothing special."
Classification:
NEUTRAL

Now classify:

Review:
{review}

Return only the classification label.


### Test Review

We will use one review as the test input for both prompts.

Using the same input makes the comparison fair.

In [8]:
review = """
The product arrived quickly and the quality is excellent.
I am very happy with my purchase.
"""

print(review)


The product arrived quickly and the quality is excellent.
I am very happy with my purchase.



### Build the Final Prompts

Both prompt templates contain the `{review}` placeholder.

We replace that placeholder with our test review.

In [9]:
zero_shot = zero_shot_prompt.format(review=review)
few_shot = few_shot_prompt.format(review=review)

print("ZERO-SHOT PROMPT")
print("=" * 50)
print(zero_shot)

print("\nFEW-SHOT PROMPT")
print("=" * 50)
print(few_shot)

ZERO-SHOT PROMPT
You are a sentiment classification assistant.

Classify the following review as exactly one of:
POSITIVE
NEGATIVE
NEUTRAL

Review:

The product arrived quickly and the quality is excellent.
I am very happy with my purchase.


Return only the classification label.

FEW-SHOT PROMPT
You are a sentiment classification assistant.

Classify the following review as exactly one of:
POSITIVE
NEGATIVE
NEUTRAL

Examples:

Review:
"I absolutely loved this movie."
Classification:
POSITIVE

Review:
"The movie was terrible and boring."
Classification:
NEGATIVE

Review:
"The movie was okay, nothing special."
Classification:
NEUTRAL

Now classify:

Review:

The product arrived quickly and the quality is excellent.
I am very happy with my purchase.


Return only the classification label.


## 4. Connect to the LLM

We will reuse the same LLM API setup from Day 06.

The API key is loaded from the `.env` file, and the OpenAI
Python client is configured to use Groq's OpenAI-compatible API.

In [12]:
import os
import openai

from dotenv import load_dotenv

load_dotenv("../.env")

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully.")

client = openai.OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

print("Client created successfully.")

API key loaded successfully.
Client created successfully.


## 5. Create a Reusable LLM Function

Instead of repeating the API call for every experiment,
we create one reusable function.

It accepts a prompt and returns the model's response.

In [15]:
def ask_llm(prompt):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [16]:
test_response = ask_llm("Say only: API connection successful.")

print(test_response)

API connection successful.


In [17]:
zero_shot_result = ask_llm(zero_shot)

print("Zero-Shot Result:")
print(zero_shot_result)

Zero-Shot Result:
POSITIVE


In [18]:
few_shot_result = ask_llm(few_shot)

print("Few-Shot Result:")
print(few_shot_result)

Few-Shot Result:
POSITIVE


## 7. Zero-Shot vs Few-Shot Comparison

Both prompts were given the same review.

The zero-shot prompt produced a classification without examples.

The few-shot prompt produced a classification using three examples.

We will now compare their results.

In [19]:
print("Zero-Shot Result:", zero_shot_result)
print("Few-Shot Result :", few_shot_result)

if zero_shot_result == few_shot_result:
    print("\nResult: Both approaches produced the same classification.")
else:
    print("\nResult: The approaches produced different classifications.")

Zero-Shot Result: POSITIVE
Few-Shot Result : POSITIVE

Result: Both approaches produced the same classification.


## 8. Comparison Summary

We can record the result of each prompting strategy.

For this test:

- Zero-shot → POSITIVE
- Few-shot → POSITIVE

The output is the same, but few-shot prompting provides examples
that can help guide the model when the classification rules are
more complicated.

In [20]:
comparison = [
    {
        "method": "Zero-Shot",
        "result": zero_shot_result
    },
    {
        "method": "Few-Shot",
        "result": few_shot_result
    }
]

comparison

[{'method': 'Zero-Shot', 'result': 'POSITIVE'},
 {'method': 'Few-Shot', 'result': 'POSITIVE'}]

## 9. Strict JSON Output

LLMs normally return natural-language text.

For applications, we often need structured data that Python can
process reliably.

For example, instead of:

The text is positive and mentions AI.

we want:

{
    "summary": "...",
    "sentiment": "positive",
    "keywords": ["AI"]
}

We will ask the model to return JSON only.

In [21]:
json_template = load_prompt("json_extraction.txt")

print(json_template)

You are an information extraction assistant.

Extract the person's information from the text.

Return ONLY valid JSON.

Required structure:

{
    "name": "string",
    "email": "string",
    "phone": "string"
}

Text:
{text}

Rules:
- Do not add extra fields.
- Do not add Markdown.
- Do not add explanations.
- If a field is missing, use null.


## 10. Test Text for JSON Extraction

We will give the model a short piece of text and ask it to
extract structured information.

In [22]:
article = """
Python is a popular programming language used for artificial
intelligence, machine learning, and data science.
Many developers use Python because it has a large ecosystem
of useful libraries.
"""

print(article)


Python is a popular programming language used for artificial
intelligence, machine learning, and data science.
Many developers use Python because it has a large ecosystem
of useful libraries.



In [24]:
json_template = load_prompt("json_extraction.txt")

In [25]:
json_prompt = json_template.format(text=article)

print(json_prompt)

You are an information extraction assistant.

Extract the person's information from the text.

Return ONLY valid JSON.

Required structure:

{
    "name": "string",
    "email": "string",
    "phone": "string"
}

Text:

Python is a popular programming language used for artificial
intelligence, machine learning, and data science.
Many developers use Python because it has a large ecosystem
of useful libraries.


Rules:
- Do not add extra fields.
- Do not add Markdown.
- Do not add explanations.
- If a field is missing, use null.


## 11. Generate JSON From the LLM

We now send the structured-output prompt to the same LLM.

The goal is to receive valid JSON that Python can parse.

In [26]:
json_response = ask_llm(json_prompt)

print(json_response)

{"name": null, "email": null, "phone": null}



that's **not strict JSON**, because of the Markdown code fences.

We'll handle/inspect that next.

---

## Step 15 — Safely parse the response

### Markdown cell

```markdown
## 12. Safely Parse JSON

We should never assume that an LLM always returns valid JSON.

We will:

1. Try to parse the response
2. Catch JSON parsing errors
3. Return `None` if parsing fails

This prevents our application from crashing when the model
returns unexpected output.

In [27]:
import json

def parse_json_safely(response):
    try:
        return json.loads(response)

    except json.JSONDecodeError as error:
        print("Invalid JSON response.")
        print("Error:", error)
        return None

In [28]:
parsed_json = parse_json_safely(json_response)

print(parsed_json)

{'name': None, 'email': None, 'phone': None}


## 10. JSON Extraction Test

Our JSON extraction prompt extracts:

- name
- email
- phone

We will provide text containing all three values and ask the LLM
to return them as structured JSON.

In [29]:
contact_text = """
My name is Rajveer Kumar.
You can contact me at rajveer@example.com.
My phone number is +91-9876543210.
"""

In [30]:
json_prompt = json_template.format(text=contact_text)

print(json_prompt)

You are an information extraction assistant.

Extract the person's information from the text.

Return ONLY valid JSON.

Required structure:

{
    "name": "string",
    "email": "string",
    "phone": "string"
}

Text:

My name is Rajveer Kumar.
You can contact me at rajveer@example.com.
My phone number is +91-9876543210.


Rules:
- Do not add extra fields.
- Do not add Markdown.
- Do not add explanations.
- If a field is missing, use null.


## 11. Generate Strict JSON

We now send the extraction prompt to the LLM.

The expected response should contain:

- name
- email
- phone

and nothing else.

In [31]:
json_response = ask_llm(json_prompt)

print(json_response)

{"name":"Rajveer Kumar","email":"rajveer@example.com","phone":"+91-9876543210"}


## 12. Safely Parse the JSON Response

We will use Python's `json.loads()` to convert the model's JSON
string into a Python dictionary.

If the model returns invalid JSON, we will catch the error instead
of allowing the notebook/application to crash.

In [32]:
import json

def parse_json_safely(response):
    try:
        data = json.loads(response)

        if not isinstance(data, dict):
            raise ValueError("Expected a JSON object.")

        return data

    except json.JSONDecodeError as error:
        print("Invalid JSON response.")
        print("Error:", error)
        return None

    except ValueError as error:
        print("Invalid response structure.")
        print("Error:", error)
        return None

In [33]:
parsed_json = parse_json_safely(json_response)

print(parsed_json)

{'name': 'Rajveer Kumar', 'email': 'rajveer@example.com', 'phone': '+91-9876543210'}


## 13. Test Invalid JSON

LLMs can sometimes return malformed JSON.

We will intentionally pass an invalid response to verify that
our parser handles the error safely.

In [34]:
bad_json = """
{
    "name": "Rajveer Kumar",
    "email": "rajveer@example.com",
    "phone":
}
"""

result = parse_json_safely(bad_json)

print("Result:", result)

Invalid JSON response.
Error: Expecting value: line 6 column 1 (char 81)
Result: None


## 14. Prompt A/B Testing

Prompt A/B testing compares two different prompt templates
using the same task and the same input.

We will:

1. Load the A/B testing prompt template
2. Create two prompt versions
3. Run both prompts
4. Compare their outputs
5. Give each output a score
6. Log the results
7. Select the winning prompt

In [35]:
ab_template = load_prompt("prompt_ab_test.txt")

print(ab_template)

You are a sentiment classification assistant.

Classify the following review as exactly one of:

POSITIVE
NEGATIVE
NEUTRAL

Review:
{review}

Return only the classification label.


## 15. Create Prompt A and Prompt B

We will compare two prompts for the same sentiment classification task.

### Prompt A — Simple

Prompt A gives the model a direct instruction.

### Prompt B — Detailed

Prompt B gives the model additional guidance and examples.

Both prompts receive exactly the same review.

In [36]:
prompt_a = ab_template

prompt_b = """
You are a sentiment classification assistant.

Classify the review as exactly one of:

POSITIVE
NEGATIVE
NEUTRAL

Use these examples as guidance:

Example 1:
Review: "I absolutely loved this product!"
Classification: POSITIVE

Example 2:
Review: "This product was terrible."
Classification: NEGATIVE

Example 3:
Review: "The product was okay, nothing special."
Classification: NEUTRAL

Now classify this review:

Review:
{review}

Return ONLY the classification label.
Do not provide an explanation.
"""

## 16. A/B Test Input

Both prompts will receive the exact same review.

This is important because we want to measure the effect of
the prompt itself.

In [37]:
ab_review = """
The product is excellent. It arrived quickly and works perfectly.
I am very satisfied with my purchase.
"""

In [38]:
final_prompt_a = prompt_a.format(review=ab_review)
final_prompt_b = prompt_b.format(review=ab_review)

print("PROMPT A")
print("=" * 60)
print(final_prompt_a)

print("\nPROMPT B")
print("=" * 60)
print(final_prompt_b)

PROMPT A
You are a sentiment classification assistant.

Classify the following review as exactly one of:

POSITIVE
NEGATIVE
NEUTRAL

Review:

The product is excellent. It arrived quickly and works perfectly.
I am very satisfied with my purchase.


Return only the classification label.

PROMPT B

You are a sentiment classification assistant.

Classify the review as exactly one of:

POSITIVE
NEGATIVE
NEUTRAL

Use these examples as guidance:

Example 1:
Review: "I absolutely loved this product!"
Classification: POSITIVE

Example 2:
Review: "This product was terrible."
Classification: NEGATIVE

Example 3:
Review: "The product was okay, nothing special."
Classification: NEUTRAL

Now classify this review:

Review:

The product is excellent. It arrived quickly and works perfectly.
I am very satisfied with my purchase.


Return ONLY the classification label.
Do not provide an explanation.



## 17. Run Prompt A

We will send Prompt A to the LLM and record its result.

In [39]:
result_a = ask_llm(final_prompt_a)

print("Prompt A result:")
print(result_a)

Prompt A result:
POSITIVE


## 18. Run Prompt B

Now we send Prompt B to the same model using the same review.

In [40]:
result_b = ask_llm(final_prompt_b)

print("Prompt B result:")
print(result_b)

Prompt B result:
POSITIVE


## 19. Evaluate the Prompts

For this experiment, we know the expected answer is:

POSITIVE

We will give each prompt:

- 1 point if the answer is correct
- 0 points if the answer is incorrect

This gives us a simple evaluation metric.